# Meeting Protocol — Prompt Testing

1. Check what production generated (prompt logs)
2. Load the same transcript locally
3. Run the prompt → get the protocol
4. Edit a prompt block, run again → compare

In [34]:
import sys, json, time, copy, requests
from pathlib import Path
from datetime import datetime

sys.path.insert(0, "..")
from utils.api_client import call_model
from utils.prompt_loader import load_prompt

PROMPTS = load_prompt("../prompts/workflow/prompt_builder.yaml")
print("Prompt blocks loaded:")
for key in PROMPTS:
    print(f"  {key} ({len(PROMPTS[key])} chars)")

Prompt blocks loaded:
  specificity_rules_de (1719 chars)
  specificity_rules_en (1482 chars)
  client_meeting_rules_de (532 chars)
  client_meeting_rules_en (479 chars)
  base_protocol_system_de (302 chars)
  base_protocol_system_en (340 chars)
  structure_analysis_de (910 chars)
  structure_analysis_en (819 chars)
  extraction_from_outline_de (859 chars)
  extraction_from_outline_en (872 chars)
  json_formatting_open_source_de (247 chars)
  json_formatting_open_source_en (180 chars)
  speaker_inference_system (1646 chars)


## 0. Production Logs

See what prompt was sent and what protocol production generated.
Use this to compare against your modified prompts.

In [35]:
WORKFLOW_URL = "https://workflow-service.mangowater-31feba87.swedencentral.azurecontainerapps.io"

response = requests.get(f"{WORKFLOW_URL}/debug/prompt-logs", params={"limit": 50}, timeout=10)

if response.status_code != 200:
    print(f"Error {response.status_code} — skip to Section 1")
else:
    logs = response.json()
    if not logs:
        print("No logs yet. Run a meeting first.")
    else:
        print(f"Found {len(logs)} entries\n")
        for i, entry in enumerate(logs):
            print(f"[{i}] {entry['pass']}  |  {entry['model']}  |  {entry['prompt_length']} chars  |  {entry['timestamp']}")
            print(f"     {entry['prompt_preview'][:120]}...")
            print()

Found 34 entries

[0] single_pass  |  gpt-4o-mini  |  7994 chars  |  2026-03-26T10:37:09.349437+00:00
     You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2....

[1] single_pass  |  gpt-4o-mini  |  102328 chars  |  2026-03-26T09:31:52.698478+00:00
     You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2....

[2] single_pass  |  gpt-4o-mini  |  50876 chars  |  2026-03-26T09:15:39.699188+00:00
     You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2....

[3] single_pass  |  gpt-4o-mini  |  102328 chars  |  2026-03-26T09:14:33.771237+00:00
     You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2....

[4] single_pass  |  gpt-4o-mini  |  50876 chars  |  2026-03-26T08:36:22.033043+00:00
     You are an expert meeting proto

In [42]:
# Pick a log entry to inspect
LOG_INDEX = 0  # <-- change this to pick a different entry

entry = logs[LOG_INDEX]
print(f"Pass: {entry['pass']}  |  Model: {entry['model']}  |  {entry['timestamp']}")
print(f"Prompt: {entry['prompt_length']} chars  |  Response: {entry['response_length']} chars")
if entry.get("usage"):
    print(f"Usage: {entry['usage']}")

# Extract speaker names from the prompt
prompt_text = entry["prompt_preview"]
prod_speakers = ""
for line in prompt_text.split("\n"):
    if line.strip().startswith("Use ONLY these names:"):
        prod_speakers = line.split(":", 1)[1].strip()
        break

print(f"Speakers: {prod_speakers}")
print(f"\n=== PRODUCTION PROMPT (system) ===")
print(prompt_text[:2000])
print(f"\n=== PRODUCTION OUTPUT (protocol) ===")
print(entry["response_preview"])

Pass: single_pass  |  Model: gpt-4o-mini  |  2026-03-26T10:37:09.349437+00:00
Prompt: 7994 chars  |  Response: 2307 chars
Usage: {'completion_tokens': 552, 'prompt_tokens': 1872, 'total_tokens': 2424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}
Speakers: Omar, Shubham

=== PRODUCTION PROMPT (system) ===
You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2. SPECIFICITY: Use concrete names, numbers, dates, and quotes from the transcript. Avoid generic descriptions.
3. Use the confirmed speaker names below, NOT the speaker IDs from the transcript.
4. Output valid JSON only.

## Speaker Name Mapping
The transcript uses speaker IDs. Replace them with these confirmed names:
- Speaker 0 / Sprecher 0 → Omar
- Speaker 1 / Sprecher 1 → Shubham

Use ONLY these names: Omar,

## 1. Load Transcript

Save the transcript from the platform as a `.txt` file in `test_cases/`.
Use the same transcript that production used, so you can compare outputs.

In [ ]:
# --- CONFIGURE THESE ---
TRANSCRIPT_FILE = "../test_cases/meeting_01.txt"
LANG = "en"              # "en" or "de"
IS_CLIENT_MEETING = False
MODEL = "gpt-4o"         # "gpt-4o" or "mistral-25b"
""
# Auto-fill speakers from production log if available
try:
    SPEAKER_NAMES = prod_speakers if prod_speakers else "Unknown"
except NameError:
    SPEAKER_NAMES = "Unknown"  # <-- set manually if you skipped Section 0

transcript = Path(TRANSCRIPT_FILE).read_text(encoding="utf-8")
print(f"{TRANSCRIPT_FILE}  |  {len(transcript)} chars  |  {LANG}  |  {MODEL}")
print(f"Speakers: {SPEAKER_NAMES}")
print("---")
print(transcript[:500] + "..." if len(transcript) > 500 else transcript)

../test_cases/meeting_01.txt  |  4812 chars  |  en  |  gpt-4o
Speakers: Omar, Shubham
---
Yes. So it started recording. I'm gonna recap what we were saying. So Shubham was showing me his output from the task of, working on the AI quality checklist, which he finished. And, basically, now we are talking about the fact that today, he is going to get onboarded to his own repo, which is called Verizon AI experiments. Right? And this is where he's gonna start modifying the prompts and seeing the difference between the, like, the what what we already currently have and the changes that he m...


## 2. Run the Prompt (Baseline)

Assembles the system prompt from blocks (same as production) and runs it.

In [ ]:
def build_system_prompt(prompts, lang, is_client=False):
    """Assemble the system prompt from blocks, same as production."""
    parts = [
        prompts[f"base_protocol_system_{lang}"],
        prompts[f"extraction_from_outline_{lang}"],
        prompts[f"specificity_rules_{lang}"],
    ]
    if is_client:
        parts.append(prompts[f"client_meeting_rules_{lang}"])
    parts.append(prompts[f"json_formatting_open_source_{lang}"])
    return "\n\n".join(parts)


system_prompt = build_system_prompt(PROMPTS, LANG, IS_CLIENT_MEETING)
user_prompt = f"Confirmed speaker names: {SPEAKER_NAMES}\n\nTRANSCRIPT:\n{transcript}"

print(f"System prompt: {len(system_prompt)} chars")
print(f"User prompt: {len(user_prompt)} chars")
print(f"\n=== SYSTEM PROMPT ===")
print(system_prompt)

System prompt: 3133 chars
User prompt: 4864 chars

=== SYSTEM PROMPT ===
Du bist ein Experte für Meeting-Protokolle.

KRITISCHE ANWEISUNGEN:
1. Antworte in derselben Sprache wie das Transkript.
2. SPEZIFITÄT: Verwende konkrete Namen, Zahlen, Daten und Zitate aus dem Transkript.
3. Verwende die bestätigten Sprechernamen, NICHT die Speaker-IDs.
4. Gib nur gültiges JSON aus.


Du bist ein erfahrener Protokollschreiber. Du erhältst zwei Eingaben:
1. Eine strukturelle Gliederung des Meetings (aus einer vorherigen Analyse)
2. Das Original-Transkript als Referenz

Deine Aufgabe: Erstelle das finale Meeting-Protokoll im vorgegebenen JSON-Format.

KRITISCHE ANWEISUNGEN:
1. SPRACHE: Antworte auf Deutsch.
2. Die Gliederung ist deine Hauptquelle für Struktur und Themen. Das Transkript ist deine Quelle für exakte Zitate, Namen und Details.
3. SPEZIFITÄT: Konkrete Namen, Zahlen, Systemnamen, Daten. Keine generischen Beschreibungen.
4. VOLLSTÄNDIGKEIT: Jedes Topic aus der Gliederung muss abgedeckt se

In [45]:
print(f"Running {MODEL}...")
start = time.time()
baseline_output = call_model(system_prompt, user_prompt, model=MODEL, max_tokens=8000)
baseline_duration = round(time.time() - start, 2)
print(f"Done in {baseline_duration}s\n")

try:
    clean = baseline_output.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    baseline_protocol = json.loads(clean)
    print(json.dumps(baseline_protocol, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(baseline_output)
    baseline_protocol = None

Running gpt-4o...
Done in 8.13s

{
  "meetingMinutes": {
    "title": "Meeting Protokoll",
    "date": "March 26",
    "participants": [
      "Omar",
      "Shubham"
    ],
    "topics": [
      {
        "topic": "Onboarding und Aufgabenverteilung",
        "discussion": "Shubham wird in sein eigenes Repository 'Verizon AI experiments' eingearbeitet. Er wird die Prompts modifizieren und verschiedene Versionen erstellen, um die beste für die Produktion auszuwählen.",
        "decisions": [
          "Shubham wird die Prompts lokal testen, bevor sie in die Produktion gehen."
        ],
        "actionItems": [
          "Shubham wird mit der Arbeit an seinem Repository beginnen und verschiedene Prompt-Versionen erstellen."
        ]
      },
      {
        "topic": "Ressourcen und Studienmaterial",
        "discussion": "Shubham hat Videos über Prompt Engineering angesehen und soll weiterhin das Papier und den Google-Leitfaden studieren, um ein besseres Verständnis zu erlangen.",
    

## 3. Edit a Prompt Block and Re-run

1. Use the cell below to see the current content of any block
2. Copy it, modify it, and run again
3. Compare the outputs in Section 4

In [ ]:
# See the current content of a block
BLOCK = "specificity_rules_de"  # <-- change to see other blocks
# Options: base_protocol_system_de, extraction_from_outline_de, specificity_rules_de,
#          client_meeting_rules_de, json_formatting_open_source_de (same for _en)

print(f"=== {BLOCK} ===")
print(PROMPTS[BLOCK])

In [ ]:
# Make a copy and override the block you want to change
PROMPTS_MODIFIED = copy.deepcopy(PROMPTS)

# Uncomment ONE and paste your modified version:

# PROMPTS_MODIFIED["specificity_rules_de"] = """
# ... your changes ...
# """

# PROMPTS_MODIFIED["extraction_from_outline_de"] = """
# ... your changes ...
# """

# PROMPTS_MODIFIED["base_protocol_system_de"] = """
# ... your changes ...
# """

changed = [k for k in PROMPTS if PROMPTS[k] != PROMPTS_MODIFIED[k]]
if changed:
    for k in changed:
        print(f"Changed: {k}  ({len(PROMPTS[k])} -> {len(PROMPTS_MODIFIED[k])} chars)")
else:
    print("Nothing changed yet -- uncomment a block above and paste your edits.")

In [ ]:
# Run with the modified prompt
modified_system = build_system_prompt(PROMPTS_MODIFIED, LANG, IS_CLIENT_MEETING)

print(f"Running {MODEL} with modified prompt...")
start = time.time()
modified_output = call_model(modified_system, user_prompt, model=MODEL, max_tokens=8000)
modified_duration = round(time.time() - start, 2)

try:
    clean = modified_output.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    modified_protocol = json.loads(clean)
    print(f"Done in {modified_duration}s\n")
    print(json.dumps(modified_protocol, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(modified_output)
    modified_protocol = None

## 4. Compare

In [ ]:
print("=" * 80)
print(f"BASELINE -- {baseline_duration}s")
print("=" * 80)
if baseline_protocol:
    print(json.dumps(baseline_protocol, indent=2, ensure_ascii=False))
else:
    print(baseline_output)

print(f"\n{'=' * 80}")
print(f"MODIFIED ({', '.join(changed) if changed else 'no changes'}) -- {modified_duration}s")
print("=" * 80)
if modified_protocol:
    print(json.dumps(modified_protocol, indent=2, ensure_ascii=False))
else:
    print(modified_output)

## 5. Save Results

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
name = Path(TRANSCRIPT_FILE).stem

results = {
    "transcript": name,
    "language": LANG,
    "model": MODEL,
    "timestamp": timestamp,
    "baseline": {
        "output": baseline_output,
        "protocol": baseline_protocol,
        "duration": baseline_duration,
    },
    "modified": {
        "output": modified_output,
        "protocol": modified_protocol,
        "duration": modified_duration,
        "changed_blocks": changed,
    },
}

outpath = Path(f"../results/{name}_{timestamp}.json")
outpath.parent.mkdir(parents=True, exist_ok=True)
outpath.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved to {outpath}")